In [1]:
import pandas as pd
import numpy as np

In [2]:
sales = pd.read_csv(
    "../data/raw/sales_train_validation.csv",
    nrows=1000
)
calendar = pd.read_csv(
    "../data/raw/calendar.csv"
)
prices = pd.read_csv(
    "../data/raw/sell_prices.csv"
)

In [3]:
print("Sales:", sales.shape)
print("Calendar:", calendar.shape)
print("Prices:", prices.shape)

Sales: (1000, 1919)
Calendar: (1969, 14)
Prices: (6841121, 4)


In [4]:
#identify identifier and sales columns
id_cols=["id","item_id","dept_id","cat_id","store_id","state_id"]

sales_cols =[
    col for col in sales.columns
    if col.startswith("d_")
]
print("Number of sales columns:",len(sales_cols))
print(sales_cols[:10])

Number of sales columns: 1913
['d_1', 'd_2', 'd_3', 'd_4', 'd_5', 'd_6', 'd_7', 'd_8', 'd_9', 'd_10']


In [16]:
# for changing the format from wide range to long we will use melt
sales_long=sales.melt(
    id_vars=id_cols,
    value_vars=sales_cols,
    var_name="d",
    value_name="sales"
)
sales_long.tail()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales
1912995,HOUSEHOLD_1_440_CA_1_validation,HOUSEHOLD_1_440,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,1
1912996,HOUSEHOLD_1_441_CA_1_validation,HOUSEHOLD_1_441,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,5
1912997,HOUSEHOLD_1_442_CA_1_validation,HOUSEHOLD_1_442,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,0
1912998,HOUSEHOLD_1_443_CA_1_validation,HOUSEHOLD_1_443,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,0
1912999,HOUSEHOLD_1_444_CA_1_validation,HOUSEHOLD_1_444,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,0


In [17]:
sales_long.shape

(1913000, 8)

In [18]:
# now we use the calendar dataframe
#first we will inspect thsi
calendar[["d","date","wm_yr_wk"]]

,d,date,wm_yr_wk
0,d_1,2011-01-29,11101
1,d_2,2011-01-30,11101
2,d_3,2011-01-31,11101
3,d_4,2011-02-01,11101
4,d_5,2011-02-02,11101
...,...,...,...
1964,d_1965,2016-06-15,11620
1965,d_1966,2016-06-16,11620
1966,d_1967,2016-06-17,11620
1967,d_1968,2016-06-18,11621


In [19]:
sales_long=sales_long.merge(
    calendar[["d","date","wm_yr_wk"]],
    on="d",
    how="left"
)
sales_long.head() 

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101


In [20]:
sales_long.shape

(1913000, 10)

In [21]:
sales_long["date"]=pd.to_datetime(sales_long["date"])

In [22]:
sales_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1913000 entries, 0 to 1912999
Data columns (total 10 columns):
 #   Column    Dtype         
---  ------    -----         
 0   id        object        
 1   item_id   object        
 2   dept_id   object        
 3   cat_id    object        
 4   store_id  object        
 5   state_id  object        
 6   d         object        
 7   sales     int64         
 8   date      datetime64[ns]
 9   wm_yr_wk  int64         
dtypes: datetime64[ns](1), int64(2), object(7)
memory usage: 146.0+ MB


In [24]:
print(sales_long.shape)
print(sales_long[["item_id", "store_id", "d", "date","wm_yr_wk", "sales"]].head(10))
print(sales_long["date"].min())#minimum date or starting date 
print(sales_long["date"].max())
print(sales_long["sales"].isna().sum())
print(sales_long["date"].isna().sum())

(1913000, 10)
         item_id store_id    d       date  wm_yr_wk  sales
0  HOBBIES_1_001     CA_1  d_1 2011-01-29     11101      0
1  HOBBIES_1_002     CA_1  d_1 2011-01-29     11101      0
2  HOBBIES_1_003     CA_1  d_1 2011-01-29     11101      0
3  HOBBIES_1_004     CA_1  d_1 2011-01-29     11101      0
4  HOBBIES_1_005     CA_1  d_1 2011-01-29     11101      0
5  HOBBIES_1_006     CA_1  d_1 2011-01-29     11101      0
6  HOBBIES_1_007     CA_1  d_1 2011-01-29     11101      0
7  HOBBIES_1_008     CA_1  d_1 2011-01-29     11101     12
8  HOBBIES_1_009     CA_1  d_1 2011-01-29     11101      2
9  HOBBIES_1_010     CA_1  d_1 2011-01-29     11101      0
2011-01-29 00:00:00
2016-04-24 00:00:00
0
0


In [25]:
# d_1 ->2011-01--29 lets check another day
sales_long[sales_long["d"]=="d_2"][["item_id", "store_id", "d", "date","wm_yr_wk", "sales"]
].head()

,item_id,store_id,d,date,wm_yr_wk,sales
1000,HOBBIES_1_001,CA_1,d_2,2011-01-30,11101,0
1001,HOBBIES_1_002,CA_1,d_2,2011-01-30,11101,0
1002,HOBBIES_1_003,CA_1,d_2,2011-01-30,11101,0
1003,HOBBIES_1_004,CA_1,d_2,2011-01-30,11101,0
1004,HOBBIES_1_005,CA_1,d_2,2011-01-30,11101,0


In [26]:
#lets inspect the price data
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [27]:
prices.shape

(6841121, 4)

In [28]:
price_key = [
    "store_id",
    "item_id",
    "wm_yr_wk"
]
#lets check do we have a composite key (duplicate) before merging the dataset
prices.duplicated(
    subset=price_key
).sum()

0

In [29]:
prices.groupby(price_key).size().value_counts().sort_index()

1    6841121
Name: count, dtype: int64

In [30]:
prices[["store_id", "item_id", "wm_yr_wk", "sell_price"]].head(10)

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26
5,CA_1,HOBBIES_1_001,11330,8.26
6,CA_1,HOBBIES_1_001,11331,8.26
7,CA_1,HOBBIES_1_001,11332,8.26
8,CA_1,HOBBIES_1_001,11333,8.26
9,CA_1,HOBBIES_1_001,11334,8.26


In [44]:
prices[
    (prices["item_id"] == "HOBBIES_1_001") &
    (prices["store_id"] == "CA_1") &
    (prices["wm_yr_wk"] == 11101)
]

,store_id,item_id,wm_yr_wk,sell_price


In [33]:
test_merge = sales_long.merge(
    prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)
test_merge

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1912995,HOUSEHOLD_1_440_CA_1_validation,HOUSEHOLD_1_440,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,1,2016-04-24,11613,14.97
1912996,HOUSEHOLD_1_441_CA_1_validation,HOUSEHOLD_1_441,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,5,2016-04-24,11613,2.77
1912997,HOUSEHOLD_1_442_CA_1_validation,HOUSEHOLD_1_442,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,0,2016-04-24,11613,8.97
1912998,HOUSEHOLD_1_443_CA_1_validation,HOUSEHOLD_1_443,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA,d_1913,0,2016-04-24,11613,2.47


In [32]:
print("Before merge:", sales_long.shape)
print("After merge:", test_merge.shape)

Before merge: (1913000, 10)
After merge: (1913000, 11)


In [36]:
print("Missing prices:", test_merge["sell_price"].isna().sum())

print(
    "Missing price percentage:",
    test_merge["sell_price"].isna().mean() * 100
)

Missing prices: 392504
Missing price percentage: 20.51772085729221


In [ ]:
# Note that this missing value and other applied on 
# our sample not on entire M5 dataset
#before filling missing values lets analyze our merged dataset 

In [37]:
#Check whether missing prices are concentrated at the beginning
missing_price = test_merge[
    test_merge["sell_price"].isna()
]

missing_price[
    ["item_id", "store_id", "date", "wm_yr_wk", "sales"]
].head(20)

,item_id,store_id,date,wm_yr_wk,sales
0,HOBBIES_1_001,CA_1,2011-01-29,11101,0
1,HOBBIES_1_002,CA_1,2011-01-29,11101,0
2,HOBBIES_1_003,CA_1,2011-01-29,11101,0
3,HOBBIES_1_004,CA_1,2011-01-29,11101,0
4,HOBBIES_1_005,CA_1,2011-01-29,11101,0
5,HOBBIES_1_006,CA_1,2011-01-29,11101,0
6,HOBBIES_1_007,CA_1,2011-01-29,11101,0
10,HOBBIES_1_011,CA_1,2011-01-29,11101,0
12,HOBBIES_1_013,CA_1,2011-01-29,11101,0
13,HOBBIES_1_014,CA_1,2011-01-29,11101,0


In [38]:
missing_price["date"].min(), missing_price["date"].max()

(Timestamp('2011-01-29 00:00:00'), Timestamp('2015-08-28 00:00:00'))

In [39]:
test_merge.groupby(
    test_merge["sell_price"].isna()
)["date"].agg(["min", "max", "count"])

,min,max,count
sell_price,,,
False,2011-01-29,2016-04-24,1520496
True,2011-01-29,2015-08-28,392504


In [40]:
#Check whether missing price rows actually have sales
missing_price.groupby(
    "sales"
).size().sort_index().head(20)

sales
0    392504
dtype: int64

In [41]:
missing_price['sales'].sum()

0

In [46]:
#Check one specific product
sample_item = missing_price.iloc[0]["item_id"]
sample_store = missing_price.iloc[0]["store_id"]

test_merge[
    (test_merge["item_id"] == sample_item) &
    (test_merge["store_id"] == sample_store)
][
    ["item_id", "store_id", "date", "wm_yr_wk", "sales", "sell_price"]
].tail(30)

,item_id,store_id,date,wm_yr_wk,sales,sell_price
1883000,HOBBIES_1_001,CA_1,2016-03-26,11609,1,8.26
1884000,HOBBIES_1_001,CA_1,2016-03-27,11609,1,8.26
1885000,HOBBIES_1_001,CA_1,2016-03-28,11609,1,8.26
1886000,HOBBIES_1_001,CA_1,2016-03-29,11609,0,8.26
1887000,HOBBIES_1_001,CA_1,2016-03-30,11609,0,8.26
1888000,HOBBIES_1_001,CA_1,2016-03-31,11609,0,8.26
1889000,HOBBIES_1_001,CA_1,2016-04-01,11609,0,8.26
1890000,HOBBIES_1_001,CA_1,2016-04-02,11610,0,8.26
1891000,HOBBIES_1_001,CA_1,2016-04-03,11610,1,8.26
1892000,HOBBIES_1_001,CA_1,2016-04-04,11610,0,8.26


In [45]:
missing_price["sales"].value_counts()
#So every single observation with a missing price has zero sales in our current dataset

sales
0    392504
Name: count, dtype: int64

In [47]:
test_merge.groupby(
    test_merge["sell_price"].isna()
)["sales"].agg(
    ["count", "sum", "min", "max"]
)

,count,sum,min,max
sell_price,,,,
False,1520496,1760081,0,294
True,392504,0,0,0


# Step 1 Validate the forecasting grain
- Does every row represent exactly one product + one store + one date?

In [48]:
#Our intended grain is:item_id + store_id + date
key_cols = ["item_id", "store_id", "date"]

duplicate_count = test_merge.duplicated(
    subset=key_cols
).sum()

duplicate_count

0

In [49]:
#We've already discovered that sell_price has missing values. Now let's verify all columns at once.
missing_values = test_merge.isna().sum().sort_values(ascending=False)

missing_values#we donot need to fix sell_price right now because in our
# current sample all Nan price have sales =0 almost except some 

sell_price    392504
id                 0
item_id            0
dept_id            0
cat_id             0
store_id           0
state_id           0
d                  0
sales              0
date               0
wm_yr_wk           0
dtype: int64

In [51]:
#checking the datatype
test_merge.dtypes

id                    object
item_id               object
dept_id               object
cat_id                object
store_id              object
state_id              object
d                     object
sales                  int64
date          datetime64[ns]
wm_yr_wk               int64
sell_price           float64
dtype: object

In [52]:
#now lets check the distribution of sales 
test_merge['sales'].describe()

count    1.913000e+06
mean     9.200633e-01
std      2.532863e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      2.940000e+02
Name: sales, dtype: float64

# Observation
- 1. count = 1,913,000 We have 1.913 million daily product-store observations in this sample.
- Remember 1000 original product/store rows × 1913 days = 1,913,000 observations.
- 2. Mean = 0.920 On average, each product-store-day observation has about:
- 0.92 units sold per day across different product in our sample
- Median That means at least half of the observations have sales of 0 units.
- This means 75% of observations have sales ≤ 1 unit.

- So most daily observations are very small.

- The distribution is therefore heavily concentrated near zero.
- 5. Maximum = 294

- Some product-store-day combinations sold as many as:

- 294 units in one day.

- That's a huge difference compared with the median of 0.This tells us the distribution is strongly right-skewed.

In [54]:
#Let's verify the exact percentage of zero-sales observations rather than relying on our earlier calculation.
zero_sales_pct = (test_merge["sales"] == 0).mean() * 100

zero_sales_pct

67.21170935703084

In [55]:
#calculate the actual count
zero_sales_count = (test_merge["sales"] == 0).sum()

zero_sales_count

1285760

In [56]:
non_zero_sales_count = (test_merge["sales"] > 0).sum()

non_zero_sales_count

627240

In [57]:
#now investigating the price distribution
test_merge["sell_price"].describe()

count    1.520496e+06
mean     5.218942e+00
std      4.333895e+00
min      1.000000e-02
25%      2.360000e+00
50%      3.970000e+00
75%      6.970000e+00
max      3.098000e+01
Name: sell_price, dtype: float64

### Price Distribution

Among observations with an available sell_price:

- Count: 1,520,496
- Mean price: 5.22
- Median price: 3.97
- Standard deviation: 4.33
- Minimum price: 0.01
- Maximum price: 30.98

The mean is higher than the median, indicating that higher-priced
observations increase the average price.

The minimum and maximum values are not automatically treated as
errors or removed as outliers. Their validity should be evaluated
using business and product context before any outlier treatment.

In [58]:
series = test_merge[
    (test_merge["item_id"] == "HOBBIES_1_001") &
    (test_merge["store_id"] == "CA_1")
].sort_values("date")
#we are checking the difference between consecutive dates
series["date"].diff().dropna().value_counts()

date
1 days    1912
Name: count, dtype: int64

In [59]:
import os

os.makedirs("../data/processed", exist_ok=True)

test_merge.to_parquet(
    "../data/processed/analytical_sample.parquet",
    index=False
)

In [60]:
pd.read_parquet(
    "../data/processed/analytical_sample.parquet"
).shape

(1913000, 11)